# Importación de librerías 📚
- Se importan las librerías necesarias para hacer las pruebas necesarias para el desarrollo de la práctica:
    - **OS:** Se utiliza para manejar *paths* (rutas o directorios).
    - **OpenCV:** Se utiliza para realizar tareas de CV (*Computer Vision*).
    - **NumPy:** Se utiliza para el manejo de *arrays*.
    - **Matplotlib:** Se utiliza para visualizar funciones e imágenes (y los cambios que realicemos sobre ellas).

In [ ]:
import sys
import os
import cv2
import numpy as np
import numpy.linalg as linalg
import matplotlib.pyplot as plt
import random

# Importar módulo Model3D
src_path = os.path.abspath(os.path.join("..", "src"))
sys.path.append(src_path)
from Model3D import Model3D

# Explicación del funcionamiento de la práctica 🧪

## Carga de datos 💾

In [ ]:
"""
- Función auxiliar para cargar imágenes
"""
def load_image(path: str):
    img = cv2.imread(path)
    if img is None:
        print(f"La imagen {path} no se pudo cargar.")
    return img

In [ ]:
"""
- Definimos los paths necesarios
"""
image_sequence_path = os.path.join("..", "data", "secuencia")
test_sequence_path = os.path.join("..", "data", "test")
models_path = os.path.join("..", "data", "3d_models")

In [ ]:
"""
- Cargamos los archivos necesarios
"""
sequence = [load_image(os.path.join(image_sequence_path, filename)) for filename in os.listdir(image_sequence_path) if filename.endswith(".jpg")]
test_sequence = [load_image(os.path.join(test_sequence_path, filename)) for filename in os.listdir(test_sequence_path) if filename.endswith(".jpg")]
models = [os.path.join(models_path, filename) for filename in os.listdir(models_path) if filename.endswith(".obj")]
template_img = load_image(os.path.join(image_sequence_path, "template_cropped.png"))
template_img_rgb = cv2.cvtColor(template_img, cv2.COLOR_BGR2RGB)
template_img_gray = cv2.cvtColor(template_img, cv2.COLOR_BGR2GRAY)

print(f"- Hay {len(sequence)} imágenes en la secuencia.")
print(f"- Hay {len(test_sequence)} imágenes en la secuencia de test.")
print(f"- Hay {len(models)} modelos 3d.")

In [ ]:
"""
- Mostramos una imagen aleatoria de la secuencia y la guardamos para hacer pruebas
"""
rand_img = random.choice(sequence)  # Obtenemos una imagen de forma aleatoria
rand_img = cv2.imread("../data/secuencia/20250314_094339.jpg")
rand_img_rgb = cv2.cvtColor(rand_img, cv2.COLOR_BGR2RGB)  # Obtenemos la imagen en RGB
rand_img_gray = cv2.cvtColor(rand_img, cv2.COLOR_BGR2GRAY)  # Obtenemos la imagen en niveles de gris

fig, axs = plt.subplots(1, 2, figsize=(12, 8))

axs[0].imshow(rand_img_rgb)
axs[0].axis("off")
axs[0].set_title("Imagen en RGB")

axs[1].imshow(rand_img_gray, cmap="gray")
axs[1].axis("off")
axs[1].set_title("Imagen en niveles de gris")

plt.show()

print(f"- Resolución: {rand_img_gray.shape[1]} px ancho x {rand_img_gray.shape[0]} px alto")

## Cálculo de la homografía ${\bf H}_t^{\text{img}}$ 🔦
- Tenemos un **objeto plano** `template_cropped.png`.
- Disponemos de una **secuencia de imágenes** que muestran el objeto plano en **diferentes posiciones**.
    - Para tomar estas imágenes, se ha utilizado una cámara cuyos **parámetros intrínsecos** vienen dados en el fichero `intrinsics.txt`.

### Detección de puntos de interés (Key Points) 🎯

- Para calcular la homografía ${\bf H}_{t}^{\text{img}}$, primero obtendremos los **P.I.** y **descriptores** de la plantilla (`template_cropped.png`) y de la imagen (`rand_img_gray`), para lo cual podemos utilizar detectores como SIFT u ORB:

In [ ]:
"""
- Función auxiliar que devuelve los P.I. y descriptores de cada píxel de la imagen "img" utilizando el detector "detector"
"""
def getKPDesc(detector, img):
    return detector.detectAndCompute(img, None)

In [ ]:
"""
- Obtener puntos de interés (key points)
"""
sift = cv2.SIFT_create()
orb = cv2.ORB_create()
kp_sift_template, desc_sift_template = getKPDesc(sift, template_img_gray)  # Query
kp_sift_img, desc_sift_img = getKPDesc(sift, rand_img_gray)  # Train
kp_orb_template, desc_orb_template = getKPDesc(orb, template_img_gray)  # Query
kp_orb_img, desc_orb_img = getKPDesc(orb, rand_img_gray)  # Train

- Utilizando un detector $\text{detector} \in \{\text{SIFT}, \text{ORB}\}$ hemos obtenido los puntos de interés $\text{P.I.}_{\text{detector}}(\text{plantilla})$ y $\text{P.I.}_{\text{detector}}(\text{imagen})$ y los descriptores $\text{DESC}_{\text{detector}}(\text{plantilla})$ y $\text{DESC}_{\text{detector}}(\text{imagen})$.

- Ahora, emparejamos los P.I. de la imagen ($\text{P.I.}_{\text{detector}}(\text{imagen})$) con los de plantilla ($\text{P.I.}_{\text{detector}}(\text{plantilla})$) en base a los descriptores obtenidos ($\text{DESC}_{\text{detector}}(\text{imagen}), \text{DESC}_{\text{detector}}(\text{plantilla})$).
    - Para ello, podemos utilizar diferentes técnicas teniendo en cuenta el detector utilizado (en nuestro caso, SIFT u ORB) y el **tipo de salida** que éste produce (en el caso de SIFT, los descriptores están en **formato de coma flotante** y en el de ORB, en **formato binario**).
    - **Brute-Force matcher:** toma una característica de $\text{DESC}_{\text{detector}}(\text{template})$ y la compara con cada característica de $\text{DESC}_{\text{detector}}(\text{imagen})$ (empleando para ello una medida de distancia). *Ver más en:* https://docs.opencv.org/4.x/d3/da1/classcv_1_1BFMatcher.html#details
        - Para SIFT utilizaremos `cv2.NORM_L2` y para ORB utilizaremos `cv2.NORM_HAMMING`.
    - **Flann Based Matcher:** utiliza una serie de algoritmos optmizizados para la búsqueda del **vecino más cercano** (funciona mejor que Brute-Force matcher para grandes cantidades de datos). *Ver más en:* https://docs.opencv.org/4.x/dc/de2/classcv_1_1FlannBasedMatcher.html#details
- Además, cabe recalcar que se empleará el filtrado de emparejamientos según enuncia **David Lowe** en su *paper* "Distinctive image features from scale-invariant keypoints" (https://scholar.google.com/citations?view_op=view_citation&hl=es&user=8vs5HGYAAAAJ&citation_for_view=8vs5HGYAAAAJ:u_35RYKgDlwC). Esta técnica consiste en 

In [ ]:
def get_good_matches(matches: tuple[tuple[cv2.DMatch]], ratio_test_threshold: float) -> list[list[cv2.DMatch]] | None:
    if len(matches) >= 4:
        good_matches = []
        for n1, n2 in matches:
            if n1.distance < (ratio_test_threshold * n2.distance):
                good_matches.append([n1])
        return good_matches
    else:
        return None

In [ ]:
"""
- Emparejamiento de P.I. utilizando BFMatcher y SIFT
"""

# Obtener matches
bf_sift = cv2.BFMatcher(normType=cv2.NORM_L2)
matches_sift_bf = bf_sift.knnMatch(desc_sift_template, desc_sift_img, k=2)

# Filtramos los matches
if matches_sift_bf:
    good_matches_sift_bf = get_good_matches(matches_sift_bf, 0.7)
    plotted_matches_sift_bf = cv2.drawMatchesKnn(template_img_rgb, kp_sift_template, rand_img_rgb, kp_sift_img, good_matches_sift_bf, None, (0, 255, 0),  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
else:
    plotted_matches_sift_bf = None

In [ ]:
"""
- Emparejamiento de P.I. utilizando BFMatcher y ORB
"""

# Obtener matches
bf = cv2.BFMatcher(normType=cv2.NORM_HAMMING)
matches_orb_bf = bf.knnMatch(desc_orb_template, desc_orb_img, k=2)

# Filtramos los matches
if matches_orb_bf:
    good_matches_orb_bf = get_good_matches(matches_orb_bf, 0.7)
    plotted_matches_orb_bf = cv2.drawMatchesKnn(template_img_rgb, kp_orb_template, rand_img_rgb, kp_orb_img, good_matches_orb_bf, None, (0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
else:
    plotted_matches_orb_bf = None

In [ ]:
"""
- Emparejamiento de P.I. utilizando FLANN y SIFT
"""

# Parámetros de FLANN recomendados por OpenCV para descriptores SIFT
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

# Obtener matches
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches_sift_flann = flann.knnMatch(desc_sift_template, desc_sift_img, k=2)

# Filtramos los matches
if matches_sift_flann:
    good_matches_sift_flann = get_good_matches(matches_sift_flann, 0.7)
    plotted_matches_sift_flann = cv2.drawMatchesKnn(template_img_rgb, kp_sift_template, rand_img_rgb, kp_sift_img, good_matches_sift_flann, None, (0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
else:
    plotted_matches_sift_flann = None

In [ ]:
"""
- Emparejamiento de P.I. utilizando FLANN y ORB
"""

# Parámetros de FLANN recomendados por OpenCV para descriptores ORB
FLANN_INDEX_LSH = 6
index_params = dict(algorithm = FLANN_INDEX_LSH,
                    table_number = 6,
                    key_size = 12,
                    multi_probe_level = 1)
search_params = dict(checks = 50)

# Obtener matches
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches_orb_flann = flann.knnMatch(desc_orb_template, desc_orb_img, k=2)

# Filtramos los matches
if matches_orb_flann:
    good_matches_orb_flann = get_good_matches(matches_orb_flann, 0.7)
    plotted_matches_orb_flann = cv2.drawMatchesKnn(template_img_rgb, kp_orb_template, rand_img_rgb, kp_orb_img, good_matches_orb_flann, None, (0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
else:
    plotted_matches_orb_flann = None

In [ ]:
"""
- Mostrar los matches obtenidos con las diferentes técnicas
"""
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

if plotted_matches_sift_bf is not None:
    axs[0, 0].imshow(plotted_matches_sift_bf)
    axs[0, 0].axis("off")
    axs[0, 0].set_title("SIFT + BFMatcher")

if plotted_matches_orb_bf is not None:
    axs[0, 1].imshow(plotted_matches_orb_bf)
    axs[0, 1].axis("off")
    axs[0, 1].set_title("ORB + BFMatcher")

if plotted_matches_sift_flann is not None:
    axs[1, 0].imshow(plotted_matches_sift_flann)
    axs[1, 0].axis("off")
    axs[1, 0].set_title("SIFT + FLANN")

if plotted_matches_orb_flann is not None:
    axs[1, 1].imshow(plotted_matches_orb_flann)
    axs[1, 1].axis("off")
    axs[1, 1].set_title("ORB + FLANN")

path_to_resources = os.path.join("..", "resources")
fig_name = "detector_matches.png"
fig.savefig(os.path.join(path_to_resources, fig_name), dpi=300, bbox_inches="tight")

plt.show()

### Obtener homografía $H_t^\text{img}$ con `cv2.findHomography` 👏

- En la figura superior podemos observar los *matches* obtenidos mediante Brute-Force matching y FLANN-Based matching para los P.I. detectados tanto con SIFT como con ORB.

- Ahora, procedemos a obtener la matriz de homografía, para lo cual utilizaremos `cv2.findHomography` y el algoritmo **RANSAC**:
    - En este contexto, RANSAC se utiliza para excluir los puntos *outliers* de los *matches* (por ejemplo, podemos observar que en ocasiones se empareja un punto del *template* con cierto punto de la mesa o de la escena en general en lugar de con el punto al que correspondería realmente).

In [ ]:
good_matches_sift_bf = [m[0] for m in good_matches_sift_bf]
good_matches_orb_bf = [m[0] for m in good_matches_orb_bf]
good_matches_sift_flann = [m[0] for m in good_matches_sift_flann]
good_matches_orb_flann = [m[0] for m in good_matches_orb_flann]

In [ ]:
def getHomography(kp1: tuple[cv2.KeyPoint], kp2: tuple[cv2.KeyPoint], matches: list[cv2.DMatch], min_good_matches: int = 4):
    if len(matches) >= min_good_matches:
        src_pts = np.array([kp1[m.queryIdx].pt for m in matches], dtype=np.float32).reshape(-1, 1, 2)
        dst_pts = np.array([kp2[m.trainIdx].pt for m in matches], dtype=np.float32).reshape(-1, 1, 2)
        return cv2.findHomography(src_pts, dst_pts, cv2.RANSAC)
    else:
        return None, None

In [ ]:
H_t_img_sift_bf, _ = getHomography(kp_sift_template, kp_sift_img, good_matches_sift_bf)
H_t_img_orb_bf, _ = getHomography(kp_orb_template, kp_orb_img, good_matches_orb_bf)
H_t_img_sift_flann, _ = getHomography(kp_sift_template, kp_sift_img, good_matches_sift_flann)
H_t_img_orb_flann, _ = getHomography(kp_orb_template, kp_orb_img, good_matches_orb_flann)

## Cálculo de la homografía ${\bf H}_w^\text{img}$ 🔦

- Para calcular la homografía ${\bf H}_w^\text{img}$, que lleva de las coordenadas respecto al SR del objeto en el mundo real en mm a las coordenadas respecto al SR de la escena, primero debemos obtener la homografía ${\bf H}_w^t$, que transforma las coordenadas respecto al SR del objeto en el mundo real en mm a las coordenadas de la plantilla (*template*) en píxeles:
    - Declaramos las 4 esquinas del objeto respecto a su propio SR en mm teniendo en cuenta los siguientes datos proporcionados por el archivo `resources/MedidasPlantilla.pdf`:
        - $\text{ancho}(\text{template}) = 210 \text{mm}$.
        - $\text{alto}(\text{template}) = 185 \text{mm}$.
    - Declaramos las 4 esquinas del objeto respecto a su propio SR en píxeles (utilizamos el atributo `shape` de NumPy).
    - Utilizamos `cv2.getPerspectiveTransform` para obtener ${\bf H}_w^t$.

In [ ]:
width_mm = 210  # Ancho de la plantilla en mm (dado en resources/MedidasPlantilla.pdf)
height_mm = 185  # Alto de la plantilla en mm (dado en resources/MedidasPlantilla.pdf)

corners_mm = np.array([
    [0, 0],  # Esquina sup. izq.
    [width_mm, 0],  # Esquina sup. der.
    [width_mm, height_mm],  # Esquina inf. der.
    [0, height_mm]  # Esquina inf. izq.
], dtype=np.float32)

corners_px = np.array([
    [0, 0],  # Esquina sup. izq.
    [template_img_gray.shape[1], 0],  # Esquina sup. der.
    [template_img_gray.shape[1], template_img_gray.shape[0]],  # Esquina inf. der.
    [0, template_img_gray.shape[0]]  # Esquina inf. izq.
], dtype=np.float32)

print("- Coordenadas de las esquinas de la plantilla en mm (SR de la plantilla):")
print(corners_mm)
print("- Coordenadas de las esquinas de la plantilla en px (SR de la plantilla):")
print(corners_px)

H_w_t = cv2.getPerspectiveTransform(corners_mm, corners_px)

- Una vez tenemos la homografía ${\bf H}_w^t$, podemos obtener la homografía ${\bf H}_w^\text{img}$ con el siguiente cálculo (multiplicación punto a punto):
$$
{\bf H}_w^\text{img} = {\bf H}_t^\text{img} \cdot {\bf H}_w^t
$$

In [ ]:
H_w_img_sift_bf = np.dot(H_t_img_sift_bf, H_w_t)
H_w_img_orb_bf = np.dot(H_t_img_orb_bf, H_w_t)
H_w_img_sift_flann = np.dot(H_t_img_sift_flann, H_w_t)
H_w_img_orb_flann = np.dot(H_t_img_orb_flann, H_w_t)

## Cómputo de la matriz de proyección $\bf P$ ⚙️
- Para poder mostrar los ejes del SR de la plantilla en la escena, mostrar modelos 3D sobre la plantilla en la escena, etc. necesitamos obtener la **matriz de proyección** $\bf P$, que transforma las coordenadas de un punto 3D respecto al SR del objeto en homogéneas a las coordenadas del punto en 2D respecto al SR de la imagen de entrenamiento.

- La matriz de proyección $\bf P$ se obtiene de la siguiente manera:
    $$
    {\bf P} = {\bf K} [{\bf R}|{\bf t}]
    $$
    - $K$ es la **matriz de parámetros intrínsecos** de la cámara con la que se tomaron las imágenes de las escenas (proporcionada en `data/secuencia/intrinsics.txt` y en `data/test/intrinsics.txt`).
    - $R$ es la **matriz de rotación** que describe la orientación del SR de la plantilla en la escena.
    - $t$ es el **vector de traslación** que indica cómo está desplazado el origen del SR de la plantilla en la escena.

- Para obtener la matriz de rotación $\bf R$ y el vector de traslación $\bf t$:
    - Obtenemos la matriz ${\bf H}^* = {\bf K}^{-1} {\bf H}_w^{\text{img}}$:
        $$
        {\bf H}^* = \left( \begin{matrix} \overbrace{h^*_{11}}^{{\bf h}_1} & \overbrace{h^*_{12}}^{{\bf h}_2} & \overbrace{h^*_{13}}^{{\bf h}_3} \\ h^*_{21} & h^*_{22} & h^*_{23} \\ h^*_{31} & h^*_{32} & h^*_{33} \end{matrix} \right)
        $$
    - Obtenemos ${\bf \lambda} = ||{\bf h}_1||$.
    - Obtenemos los vectores necesarios para construir la matriz de rotación:
        $$
        \begin{align*}
        {\bf r}_1 &= \frac{{\bf h}_1}{{\bf \lambda}} \\
        {\bf r}_2 &= \frac{{\bf h}_2}{{\bf \lambda}} \\
        {\bf r}_3 &= {\bf r}_1 \times {\bf r}_2
        \end{align*}
        $$
    - Obtenemos el vector de traslación $\bf t$:
        $$
        {\bf t} = \frac{{\bf h}_3}{{\bf \lambda}}
        $$

In [ ]:
K = np.loadtxt(os.path.join(image_sequence_path, "intrinsics.txt"))
H_star_sift_bf = linalg.inv(K) @ H_w_img_sift_bf
H_star_orb_bf = linalg.inv(K) @ H_w_img_orb_bf
H_star_sift_flann = linalg.inv(K) @ H_w_img_sift_flann
H_star_orb_flann = linalg.inv(K) @ H_w_img_orb_flann

In [ ]:
def getPMatrix(intrinsics: np.ndarray, homography: np.ndarray):
    h1 = homography[:, 0]
    h2 = homography[:, 1]
    h3 = homography[:, 2]

    lambda_val = linalg.norm(h1)

    r1 = h1 / lambda_val
    r2 = h2 / lambda_val
    r3 = np.cross(r1, r2)
    t = (h3 / lambda_val).reshape(-1, 1)

    R = np.vstack((r1, r2, r3)).T
    
    return intrinsics @ np.hstack((R, t)), R, t

In [ ]:
P_sift_bf, R_sift_bf, t_sift_bf = getPMatrix(K, H_star_sift_bf)
P_orb_bf, R_orb_bf, t_orb_bf = getPMatrix(K, H_star_orb_bf)
P_sift_flann, R_sift_flann, t_sift_flann = getPMatrix(K, H_star_sift_flann)
P_orb_flann, R_orb_flann, t_orb_flann = getPMatrix(K, H_star_orb_flann)

## Realidad aumentada: proyección del SR sobre el objeto en la escena 📸
- Finalmente, teniendo la matriz de proyección $\bf P$, podemos obtener las coordenadas de las rectas que componen la visualización del SR en las escenas:
    - Establecemos la longitud de los ejes (`axs_length`). Para simplificar, llamaremos $l$ a esta longitud.
    - Definimos 4 puntos en **coordenadas homogéneas**:
        - Origen de coordenadas: $\left( \begin{matrix} 0 & 0 & 0 & 1 \end{matrix} \right)^\top$.
        - Punto en el eje X: $\left( \begin{matrix} l & 0 & 0 & 1 \end{matrix} \right)^\top$.
        - Punto en el eje Y: $\left( \begin{matrix} 0 & l & 0 & 1 \end{matrix} \right)^\top$.
        - Punto en el eje Z: $\left( \begin{matrix} 0 & 0 & l & 1 \end{matrix} \right)^\top$.

In [ ]:
def get_points2D(points3D: np.ndarray, P: np.ndarray) -> np.ndarray:
    points2D = P @ points3D
    return points2D / points2D[2, :]

In [ ]:
axs_length = 100
points3D = np.array([
    [0, 0, 0, 1],
    [axs_length, 0, 0, 1],
    [0, axs_length, 0, 1],
    [0, 0, -axs_length, 1]
], dtype=np.float32).T

points2D_sift_bf = get_points2D(points3D, P_sift_bf)
points2D_orb_bf = get_points2D(points3D, P_orb_bf)
points2D_sift_flann = get_points2D(points3D, P_sift_flann)
points2D_orb_flann = get_points2D(points3D, P_orb_flann)

In [ ]:
def plotSR(img: np.ndarray, origin: np.ndarray, xpoint: np.ndarray, ypoint: np.ndarray, zpoint: np.ndarray):
    origin_int = (int(origin[0]), int(origin[1]))
    xpoint_int = (int(xpoint[0]), int(xpoint[1]))
    ypoint_int = (int(ypoint[0]), int(ypoint[1]))
    zpoint_int = (int(zpoint[0]), int(zpoint[1]))
    plt.imshow(img)
    plt.plot((origin_int[0], xpoint_int[0]), (origin_int[1], xpoint_int[1]), color="red", linewidth=3)
    plt.plot((origin_int[0], ypoint_int[0]), (origin_int[1], ypoint_int[1]), color="green", linewidth=3)
    plt.plot((origin_int[0], zpoint_int[0]), (origin_int[1], zpoint_int[1]), color="blue", linewidth=3)
    plt.axis("off")
    plt.show()

In [ ]:
"""
- Proyección del SR (SIFT + BF)
"""
points2D_cart = points2D_sift_flann[:2, :].T
origin = points2D_cart[0]
xpoint = points2D_cart[1]
ypoint = points2D_cart[2]
zpoint = points2D_cart[3]
plotSR(rand_img_rgb, origin, xpoint, ypoint, zpoint)

In [ ]:
"""
- Proyección del SR (ORB + BF)
"""
points2D_cart = points2D_orb_bf[:2, :].T
origin = points2D_cart[0]
xpoint = points2D_cart[1]
ypoint = points2D_cart[2]
zpoint = points2D_cart[3]
plotSR(rand_img_rgb, origin, xpoint, ypoint, zpoint)

In [ ]:
"""
- Proyección del SR (SIFT + FLANN)
"""
points2D_cart = points2D_sift_flann[:2, :].T
origin = points2D_cart[0]
xpoint = points2D_cart[1]
ypoint = points2D_cart[2]
zpoint = points2D_cart[3]
plotSR(rand_img_rgb, origin, xpoint, ypoint, zpoint)

In [ ]:
"""
- Proyección del SR (ORB + FLANN)
"""
points2D_cart = points2D_orb_flann[:2, :].T
origin = points2D_cart[0]
xpoint = points2D_cart[1]
ypoint = points2D_cart[2]
zpoint = points2D_cart[3]
plotSR(rand_img_rgb, origin, xpoint, ypoint, zpoint)

## Realidad aumentada: cubo 3D
- Ahora, se nos requiere "colocar" un modelo (`.obj`) de cubo tridimensional de **43 mm de lado** en la escena sobre el cuadrado azul claro del objeto plano.
- Las esquinas de la región del objeto plano sobre la que deberá colocarse el cubo (el cuadrado azul claro) tiene las siguientes coordenadas respecto al SR del objeto en milímetros:
    - La **esquina superior izquierda** se encuentra en el punto $\left( \begin{matrix} 105 & 71 & 1 \end{matrix} \right)^\top$.
    - La **esquina superior derecha** se encuentra en el punto $\left( \begin{matrix} 148 & 71 & 1 \end{matrix} \right)^\top$.
    - La **esquina inferior derecha** se encuentra en el punto $\left( \begin{matrix} 148 & 114 & 1 \end{matrix} \right)^\top$.
    - La **esquina inferior izquierda** se encuentra en el punto $\left( \begin{matrix} 105 & 114 & 1 \end{matrix} \right)^\top$.

In [ ]:
"""
- Definimos los vértices de la región
"""
points2D_mm_hom = np.array([
    [105, 71, 1],
    [148, 71, 1],
    [148, 114, 1],
    [105, 114, 1]
], dtype=np.float32).T

points3D_mm_hom = np.vstack((points2D_mm_hom[:2, :], np.zeros(points2D_mm_hom.shape[1]), points2D_mm_hom[2, :]))

print("- Coordenadas 2D (homogéneas) de los vértices de la región en mm:")
print(points2D_mm_hom)

print("- Coordenadas 3D (homogéneas) de los vértices de la región en mm:")
print(points3D_mm_hom)

points2D_px_hom = H_w_t @ points2D_mm_hom
points2D_px_hom /= points2D_px_hom[2, :]

print("- Coordenadas (homogéneas) de los vértices de la región en px:")
print(points2D_px_hom)

plt.imshow(template_img_rgb)
plt.scatter(points2D_px_hom[0, :], points2D_px_hom[1, :], color="red", marker='x')
plt.title("Vértices de la región de la plantilla")
plt.axis("off")
plt.show()

In [ ]:
"""
- Cargamos el modelo ".obj" con la clase Model3D
"""
model = Model3D()
model.load_from_obj(random.choice(models))  # Elegimos un modelo 3D de forma aleatoria

print("- Coordenadas 3D de los vértices del cubo:")
print(model.vertices)

- Como podemos observar, el cubo se encuentra centrado en $\left( \begin{matrix} 0 & 0 & 0 \end{matrix} \right)^\top$ y la distancia entre los vértices consecutivos es de 2 unidades.

- Gracias a esta información, podemos obtener la escala de la siguiente manera:

In [ ]:
"""
- Calculamos la escala y escalamos el cubo con el método `scale()`
"""
cube_scale = 43 / 2
print(f"- La escala es: {cube_scale}")
model.scale(cube_scale)
print("- Vértices del cubo tras escalarlo:")
print(model.vertices)

- También debemos trasladar el modelo 3D a su posición correcta. Para ello, debemos tener en cuenta que el cubo tiene el centro de su base en el punto $\left( \begin{matrix} 0 & 0 & -1 \end{matrix} \right)$, pero, como lo hemos escalado, ahora se encuentra en $\left( \begin{matrix} 0 & 0 & -21.5 \end{matrix} \right)$.
- Tenemos que el centro de la región de la plantilla se encuentra en $\left( \begin{matrix} (148+105)/2 & (114+71)/2 & 0 \end{matrix} \right) = \left( \begin{matrix} 126.5 & 92.5 & 0 \end{matrix} \right)$ (en mm respecto al SR de la plantilla).
- Teniendo en cuenta la posición del centro de la base del cubo respecto al SR del cubo y la posición del centro de la región de la plantilla sobre la que se colocará el cubo respecto al SR de la plantilla, concluimos que la traslación a aplicar es:
$$
t = \left( \begin{matrix} (126.5 + 0) & (92.5 + 0) & (0 + 21.5) \end{matrix} \right) = \left( \begin{matrix} 126.5 & 92.5 & 21.5 \end{matrix} \right)
$$

In [ ]:
"""
- Definimos el vector de traslación "t" y trasladamos el cubo con el método `translate`
"""
cube_t = np.array([126.5, 92.5, -21.5]).reshape(1, 3)
print("- Traslación a aplicar a los vértices del cubo:")
print(cube_t)
model.translate(cube_t)
print("- Vértices del cubo tras trasladarlo:")
print(model.vertices)

- Finalmente, proyectamos el cubo sobre la imagen de la escena mediante el método `plot_on_image()` utilizando las matrizes de proyección obtenidas con los diferentes métodos anteriormente:

In [ ]:
"""
- Con la matriz de proyección obtenida con SIFT y BF-matcher
"""
cube_img_sift_bf = rand_img_rgb.copy()
model.plot_on_image(cube_img_sift_bf, P_sift_bf)
plt.imshow(cube_img_sift_bf)
plt.title("Proyección del cubo con SIFT y BF-matcher")
plt.axis("off")
plt.show()

In [ ]:
"""
- Con la matriz de proyección obtenida con ORB y BF-matcher
"""
cube_img_orb_bf = rand_img_rgb.copy()
model.plot_on_image(cube_img_orb_bf, P_orb_bf)
plt.imshow(cube_img_orb_bf)
plt.title("Proyección del cubo con ORB y BF-matcher")
plt.axis("off")
plt.show()

In [ ]:
"""
- Con la matriz de proyección obtenida con SIFT y FLANN
"""
cube_img_sift_flann = rand_img_rgb.copy()
model.plot_on_image(cube_img_sift_flann, P_sift_flann)
plt.imshow(cube_img_sift_flann)
plt.title("Proyección del cubo con SIFT y FLANN")
plt.axis("off")
plt.show()

In [ ]:
"""
- Con la matriz de proyección obtenida con ORB y FLANN
"""
cube_img_orb_flann = rand_img_rgb.copy()
model.plot_on_image(cube_img_orb_flann, P_orb_flann)
plt.imshow(cube_img_orb_flann)
plt.title("Proyección del cubo con ORB y FLANN")
plt.axis("off")
plt.show()

## Cálculo de la homografía ${\bf H}_t^{\text{img}}$ utilizando el marco rojo 🔦

### Obtener el contorno en la plantilla

In [ ]:
def get_best_contour(contours, epsilon: float = 0.02):
    for contour in contours:
        perimeter = cv2.arcLength(contour, True)
        epsilon_val = epsilon * perimeter
        approx_poly = cv2.approxPolyDP(contour, epsilon_val, True)

        if len(approx_poly) == 4 and cv2.isContourConvex(approx_poly):
            return approx_poly
    return None

In [ ]:
template_img_r = template_img_rgb[:, :, 0].copy()
mask_inner_template = np.array(template_img_r > 250, dtype=np.uint8)
mask_outer_template = np.array(template_img_r < 250, dtype=np.uint8)

contours_inner_template, _ = cv2.findContours(mask_inner_template, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours_outer_template, _ = cv2.findContours(mask_outer_template, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"- Se han detectado {len(contours_inner_template)} contornos interiores y {len(contours_outer_template)} exteriores.")
contour_inner_template = get_best_contour(contours_inner_template)
contour_outer_template = get_best_contour(contours_outer_template)

plt.imshow(template_img_rgb)
plt.scatter(contour_inner_template[:, 0, 0], contour_inner_template[:, 0, 1], color="blue", linewidth=8)
plt.scatter(contour_outer_template[:, 0, 0], contour_outer_template[:, 0, 1], color="blue", linewidth=8)

plt.axis("off")
plt.show()

### Preprocesado

In [ ]:
def get_red_mask_rgb(img_rgb, r_thresh=140, g_thresh=110, b_thresh=110):
    r = img_rgb[:, :, 0]
    g = img_rgb[:, :, 1]
    b = img_rgb[:, :, 2]
    mask = (r > r_thresh) & (g < g_thresh) & (b < b_thresh)
    return mask.astype(np.uint8) * 255

def get_red_mask_hsv(img_rgb, low1=(0, 30, 30), high1=(15, 255, 255), 
                     low2=(150, 30, 30), high2=(179, 255, 255)):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    mask1 = cv2.inRange(hsv, np.array(low1), np.array(high1))
    mask2 = cv2.inRange(hsv, np.array(low2), np.array(high2))
    return cv2.bitwise_or(mask1, mask2)

def adaptive_threshold(img_one_channel, block_size=31, C=10):
    return cv2.adaptiveThreshold(
        img_one_channel,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY,
        block_size,
        C
    )

def get_mser_mask(img_gray, min_area=1000):
    mser = cv2.MSER_create()
    regions, _ = mser.detectRegions(img_gray)
    mask = np.zeros_like(img_gray, dtype=np.uint8)
    for region in regions:
        hull = cv2.convexHull(region.reshape(-1, 1, 2))
        area = cv2.contourArea(hull)
        if area > min_area:
            cv2.drawContours(mask, [hull], -1, 255, -1)
    return mask

# Morfológicas 
def apply_close(mask, kernel_size=(5, 15)):
    kernel = np.ones(kernel_size, np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

def apply_open(mask, kernel_size=(3, 3)):
    kernel = np.ones(kernel_size, np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

def apply_erode(mask, kernel_size=(3, 3), iterations=1):
    kernel = np.ones(kernel_size, np.uint8)
    return cv2.erode(mask, kernel, iterations=iterations)

def apply_dilate(mask, kernel_size=(3, 3), iterations=1):
    kernel = np.ones(kernel_size, np.uint8)
    return cv2.dilate(mask, kernel, iterations=iterations)

In [ ]:
# Aplicar pipeline: Adaptive Threshold + Erosión + Apertura + Inversión
rand_img_r = rand_img_rgb[:,:,0]
hsv_mask = get_red_mask_hsv(rand_img_rgb)
adaptive_mask = adaptive_threshold(rand_img_r, block_size=31, C=8)
eroded_mask = apply_erode(adaptive_mask, kernel_size=(2, 2), iterations=1)
opened_mask = apply_open(eroded_mask)
inverted_mask = 255 - opened_mask

In [ ]:
masks = [rand_img_r, hsv_mask, adaptive_mask, eroded_mask, opened_mask, inverted_mask ]
titles = ["Canal Rojo", 'Red HSV Mask',"Umbralizado ", "Erosión", "Abierta", "Invertida"]

fig, axs = plt.subplots(1, 6, figsize=(20, 4))
for i in range(6):
    axs[i].imshow(masks[i], cmap="gray")
    axs[i].set_title(titles[i])
    axs[i].axis("off")
plt.tight_layout()
plt.show()


### Jerarquía de contornos

In [ ]:
def is_a4_ratio(quad, tolerance=0.2):
    """
    Comprueba si el contorno tiene proporción A4 (≈1.41) con cierta tolerancia.
    """
    rect = cv2.minAreaRect(quad)
    width, height = rect[1]
    if width == 0 or height == 0:
        return False
    ratio = max(width, height) / min(width, height)
    
    return abs(ratio - 1.41) < tolerance

def find_nested_quads_by_hierarchy(contours, hierarchy, min_area=10000):
    """
    Devuelve pares de cuadriláteros donde uno está dentro de otro, usando la jerarquía.
    """
    # hierarchy da info de la jerarquía de contornos: [Next, Previous, First_Child, Parent]
    
    nested_pairs = []

    for idx, h in enumerate(hierarchy[0]):
        parent_idx = h[3]
        if parent_idx != -1:
            child = contours[idx]
            parent = contours[parent_idx]

            if (cv2.contourArea(child) > min_area and 
                cv2.contourArea(parent) > min_area):

                approx_child = cv2.approxPolyDP(child, 0.02 * cv2.arcLength(child, True), True)
                approx_parent = cv2.approxPolyDP(parent, 0.02 * cv2.arcLength(parent, True), True)

                if (len(approx_child) == 4 and cv2.isContourConvex(approx_child) and
                    len(approx_parent) == 4 and cv2.isContourConvex(approx_parent)):

                    # Ignorar si alguno tiene forma de folio
                    if is_a4_ratio(approx_parent) or is_a4_ratio(approx_child):
                       continue

                    nested_pairs.append((approx_parent, approx_child))

    return nested_pairs

def order_corners(quad):
    """
    Ordena consistentemente los puntos del cuadrilátero: top-left, top-right, bottom-right, bottom-left
    """
    pts = quad.reshape(4, 2)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)

    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = pts[np.argmin(s)]        # top-left
    ordered[2] = pts[np.argmax(s)]        # bottom-right
    ordered[1] = pts[np.argmin(diff)]     # top-right
    ordered[3] = pts[np.argmax(diff)]     # bottom-left
    return ordered

def filter_similar_quad_pairs(pairs, max_corner_dist=10):
    """
    Elimina pares (outer, inner) si ambos cuadriláteros son prácticamente iguales,
    comparando la distancia entre sus esquinas.
    
    Args:
        pairs: lista de tuplas (outer_quad, inner_quad)
        max_corner_dist: umbral máximo de distancia media entre esquinas para considerar duplicados

    Return:
        Lista de pares filtrados
    """
    filtered = []
    for outer, inner in pairs:
        
        outer_ord = order_corners(outer)
        inner_ord = order_corners(inner)

        distances = np.linalg.norm(outer_ord - inner_ord, axis=1)
        mean_dist = np.mean(distances)

        if mean_dist > max_corner_dist:
            filtered.append((outer, inner))

    return filtered

In [ ]:
# Buscamos los contornos
# 'hierarchy' da info sobre la jerarquía de contornos: [Next, Previous, First_Child, Parent]
contours, hierarchy=  cv2.findContours(inverted_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
nested_quads = find_nested_quads_by_hierarchy(contours,hierarchy)

- Filtramos los pares cuyos contornos exterior e interior sean muy similares comparando la distancia entre una esquina. (También se consideró utilizar **Intersection over Union** por ser más robusto y flexible para otros proyectos,pero no se realizó por falta de tiempo).

In [ ]:
# Mostramos los los pares cuyos contornos exterior e interior son muy similares
filtered_nested_quads=filter_similar_quad_pairs(nested_quads)
imagen= rand_img.copy()
outer, inner = filtered_nested_quads[0]  # Un solo par
cv2.drawContours(imagen, [outer], -1, (0, 255, 0), 5)  # Verde: exterior
cv2.drawContours(imagen, [inner], -1, (255, 0, 0), 5)  # Rojo: interior

plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
plt.title("Par filtrado")
plt.axis("off")
plt.show()

In [ ]:
# Dibujar esquinas numeradas
img=rand_img.copy()

# Ordenar esquinas
outer_ord = order_corners(outer)
inner_ord = order_corners(inner)

plt.figure(figsize=(8, 8))
plt.imshow(rand_img_rgb)

# Esquinas exteriores en verde con etiquetas
for i, pt in enumerate(outer_ord):
    plt.scatter(pt[0], pt[1], color="green", s=30)
    plt.text(pt[0]+10, pt[1], f"{i}", color="green", fontsize=12)

# Esquinas interiores en rojo con etiquetas
for i, pt in enumerate(inner_ord):
    plt.scatter(pt[0], pt[1], color="blue", s=30)
    plt.text(pt[0]+10, pt[1], f"{i}", color="blue", fontsize=12)

plt.title("Esquinas ordenadas: exterior (verde), interior (azul)")
plt.axis("off")
plt.show()

### Obtenemos la homografía $H_t^\text{img}$

In [ ]:
contour_outer_template = contour_outer_template.astype(np.float32)
H_t_img2, _ = cv2.findHomography(contour_outer_template, outer_ord, cv2.RANSAC)